# 02 · Approach 1: LLM Caption → Dense + FTS

**Strategy:** Use Claude to generate rich scene descriptions from the comic-transcriber output, then embed those descriptions with Pinecone's integrated `llama-text-embed-v2`. Dialogue text is stored as a full-text search (FTS) field for exact keyword retrieval.

**Namespace:** `ns-text-caption`

**Blog post insight:** LLM captions add semantic richness — a query for *"a moment of betrayal"* can surface relevant pages even when no character says that word. FTS on dialogue nails exact quotes and sound effects.

---

## When to use this approach
- You want semantic search over *meaning* (themes, mood, setting)
- Your source content is primarily visual with little usable text metadata
- You want keyword search on transcribed dialogue/captions as a complementary signal

In [ ]:
import json
import os
import time
from pathlib import Path

import anthropic
from dotenv import load_dotenv
from pinecone import Pinecone
from pinecone.preview import SchemaBuilder
from tqdm import tqdm

load_dotenv()

TRANSCRIPTS_DIR = Path("data/transcripts")
INDEX_NAME      = "comics-multimodal"
NAMESPACE       = "ns-text-caption"
BATCH_SIZE      = 50

pc     = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
claude = anthropic.Anthropic()

## Create (or reuse) the hybrid index

This index is shared across all three approaches. It has dense + sparse + FTS fields in a single index via `SchemaBuilder`. Run this cell once — it handles the 409 case if the index already exists.

In [ ]:
schema = (
    SchemaBuilder()
    .add_string_field(
        name="dialogue",
        full_text_search={"language": "en", "stemming": True, "stop_words": True},
    )
    .add_string_field(name="comic_title", filterable=True)
    .add_string_field(name="page_id", filterable=True)
    .add_dense_vector_field(name="embedding", dimension=1024, metric="dotproduct")
    .add_sparse_vector_field(name="sparse_embedding")
    .build()
)

try:
    index_model = pc.preview.indexes.create(
        name=INDEX_NAME,
        schema=schema,
        read_capacity={"mode": "OnDemand"},
    )
    print(f"Created index: {INDEX_NAME}")
except Exception as e:
    if "409" in str(e) or "already exists" in str(e).lower():
        print(f"Index '{INDEX_NAME}' already exists — reusing.")
    else:
        raise

# Wait for ready
while True:
    info = pc.preview.indexes.describe(name=INDEX_NAME)
    if info.status.ready:
        INDEX_HOST = info.host
        break
    print("  waiting for index to be ready...")
    time.sleep(3)

print(f"Index ready. Host: {INDEX_HOST}")
index = pc.index(host=INDEX_HOST)

## Generate LLM scene captions

We use `claude-haiku-4-5-20251001` (fast and cheap) to generate a rich scene description from the comic-transcriber transcript. The prompt is included verbatim so you can reproduce or tune it.

In [ ]:
CAPTION_PROMPT = """\
You are describing a comic book page for a semantic search index.
Based on the transcript below, write a dense 2–4 sentence scene description covering:
- Setting and time of day
- Characters present and their emotional state
- Key action or plot moment happening
- Visual mood or atmosphere

Be specific. Do NOT include meta-commentary about the comic itself.
Write only the description, no preamble.

TRANSCRIPT:
{transcript}"""


def generate_caption(record: dict) -> str:
    transcript = record["raw_transcript"][:3000]  # stay well within context
    msg = claude.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=256,
        messages=[{"role": "user", "content": CAPTION_PROMPT.format(transcript=transcript)}],
    )
    return msg.content[0].text.strip()

In [ ]:
records = [json.loads(p.read_text()) for p in sorted(TRANSCRIPTS_DIR.glob("*.json"))]
print(f"Loaded {len(records)} transcripts")

# Generate captions (cached in each record's JSON to avoid re-billing)
for record in tqdm(records, desc="Generating captions"):
    if record.get("llm_caption"):
        continue
    record["llm_caption"] = generate_caption(record)
    path = TRANSCRIPTS_DIR / f"{record['page_id']}.json"
    path.write_text(json.dumps(record, ensure_ascii=False, indent=2))

print("Captions ready.")

## Upsert to Pinecone (ns-text-caption)

We pass the LLM caption as the `embedding` text field — Pinecone's integrated `llama-text-embed-v2` inference embeds it automatically. Dialogue goes into the FTS `dialogue` field.

In [ ]:
documents = [
    {
        "_id":        r["page_id"],
        "embedding":  r["llm_caption"],   # integrated inference embeds this
        "dialogue":   r["all_dialogue"],
        "comic_title": r["comic_title"],
        "page_id":    r["page_id"],
        "page_num":   r["page_num"],
        "file_path":  r["file_path"],
        "llm_caption": r["llm_caption"],
    }
    for r in records
]

index.documents.batch_upsert(
    namespace=NAMESPACE,
    documents=documents,
    batch_size=BATCH_SIZE,
    show_progress=True,
)
print(f"Upserted {len(documents)} pages to {NAMESPACE}.")

## Search demos

In [ ]:
from IPython.display import display
from PIL import Image


def show_results(matches, label=""):
    if label:
        print(f"\n{'='*60}\n{label}\n{'='*60}")
    for m in matches:
        caption = getattr(m, "llm_caption", "") or ""
        print(f"[{m._score:.3f}] {m._id} — {caption[:120]}")
        fp = getattr(m, "file_path", None)
        if fp and Path(fp).exists():
            img = Image.open(fp)
            img.thumbnail((250, 350))
            display(img)

In [ ]:
# Dense semantic search on LLM captions
resp = index.documents.search(
    namespace=NAMESPACE,
    top_k=3,
    score_by=[{"type": "dense_vector", "field": "embedding", "query": "hero discovers villain's secret lair"}],
    include_fields=["page_id", "comic_title", "llm_caption", "file_path"],
)
show_results(resp.matches, "Dense semantic: 'hero discovers villain's secret lair'")

In [ ]:
# FTS on dialogue
resp = index.documents.search(
    namespace=NAMESPACE,
    top_k=3,
    score_by=[{"type": "text", "fields": ["dialogue"], "query": "look out"}],
    include_fields=["page_id", "comic_title", "dialogue", "file_path"],
)
show_results(resp.matches, "FTS on dialogue: 'look out'")

In [ ]:
# Filter by keyword in dialogue, rank by dense vector semantics
resp = index.documents.search(
    namespace=NAMESPACE,
    top_k=3,
    score_by=[{"type": "dense_vector", "field": "embedding", "query": "tense confrontation"}],
    filter={"dialogue": {"$match_any": ["help", "stop", "run"]}},
    include_fields=["page_id", "comic_title", "llm_caption", "file_path"],
)
show_results(resp.matches, "Keyword filter + dense rerank: 'tense confrontation' where dialogue contains help/stop/run")